# Bhagavad Gita Knowledge Graph — Loader

Loads chapters, verses, speakers, addressees, epithets, setting, and lemmatized terms
from the English translations into the local Neo4j `TheGitaProject` database.

- Deterministic + idempotent: re-running rebuilds the graph with no duplicates.
- Requires a local Neo4j with a `TheGitaProject` database and a `gita-knowledge-graph/.env`
  (copy from `.env.example`).

In [1]:
import collections
import sys
from pathlib import Path

import spacy
from dotenv import load_dotenv
from neo4j import GraphDatabase


def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    return start


ROOT = find_repo_root(Path.cwd())
PKG = ROOT / "gita-knowledge-graph"
sys.path.insert(0, str(PKG))  # make gita_kg importable regardless of cwd

import gita_kg as gk

## 1. Config & connect

In [2]:
load_dotenv(PKG / ".env", override=True)  # .env is the source of truth
cfg = gk.load_config()
driver = GraphDatabase.driver(cfg.uri, auth=(cfg.user, cfg.password))
driver.verify_connectivity()
print("connected:", cfg.uri, "->", cfg.database)

connected: bolt://localhost:7687 -> neo4j


## 2. spaCy pipeline (EntityRuler for epithets)

In [3]:
nlp = gk.build_epithet_ruler(spacy.load("en_core_web_sm"))

## 3. Parse the verses

In [4]:
VERSES_DIR = ROOT / "data/TheGitaProject/Verses"
records = gk.build_records(VERSES_DIR, nlp)
print(f"parsed {len(records)} verses")
print("speakers:", collections.Counter(r.speaker for r in records))

parsed 701 verses
speakers: Counter({'Krishna': 574, 'Arjuna': 86, 'Sanjaya': 40, 'Dhritarashtra': 1})


## 4. Load into Neo4j (constraints → seeds → verses)

In [5]:
def run_ops(ops):
    with driver.session(database=cfg.database) as session:
        for cypher, params in ops:
            session.run(cypher, **params)


run_ops(gk.constraint_ops())
run_ops(gk.seed_ops())
run_ops(gk.verse_ops(records))
print("load complete")

load complete


## 5. Verification

In [6]:
def one(cypher):
    with driver.session(database=cfg.database) as s:
        return s.run(cypher).single()[0]


print("verses:", one("MATCH (v:Verse) RETURN count(v)"))
print(
    "no SPOKEN_BY:",
    one("MATCH (v:Verse) WHERE NOT (v)-[:SPOKEN_BY]->() RETURN count(v)"),
)
print(
    "no ADDRESSED_TO:",
    one("MATCH (v:Verse) WHERE NOT (v)-[:ADDRESSED_TO]->() RETURN count(v)"),
)
print("terms:", one("MATCH (t:Term) RETURN count(t)"))
print("epithet edges:", one("MATCH ()-[r:USES_EPITHET]->() RETURN count(r)"))

verses: 701
no SPOKEN_BY: 0
no ADDRESSED_TO: 0
terms: 1171
epithet edges: 32


In [7]:
with driver.session(database=cfg.database) as s:
    rows = s.run(
        "MATCH (v:Verse)-[:SPOKEN_BY]->(:Person {name:'Arjuna'}) "
        "RETURN v.id AS id ORDER BY v.chapter, v.verse LIMIT 10"
    ).values()
print("Arjuna's first verses:", rows)

Arjuna's first verses: [['1.21'], ['1.22'], ['1.23'], ['1.28'], ['1.29'], ['1.30'], ['1.31'], ['1.32'], ['1.33'], ['1.34']]


In [8]:
with driver.session(database=cfg.database) as s:
    rows = s.run(
        "MATCH (:Chapter {number:2})-[:HAS_VERSE]->(v)-[m:MENTIONS_TERM]->(t) "
        "RETURN t.lemma AS term, sum(m.count) AS n ORDER BY n DESC LIMIT 10"
    ).values()
print("Chapter 2 top terms:", rows)

Chapter 2 top terms: [['wisdom', 13], ['say', 13], ['sens', 10], ['pleasure', 9], ['grieve', 9], ['attain', 9], ['speak', 9], ['desire', 9], ['man', 9], ['word', 8]]


## 6. Sanskrit grounding

Ground each verse in the source language: store its **Original Sanskrit** and
**Transliteration**, then parse the per-verse **Word Meanings** table — dropping
particles/pronouns/epithets/speaker-markers and normalizing surface forms to
roots — and load `SanskritTerm` nodes with `CONTAINS_TERM` edges.


In [9]:
# Ground each verse in its Sanskrit Word Meanings vocabulary
sanskrit_by_id = {}
for f in sorted(VERSES_DIR.rglob("*.md")):
    if "Verse" not in f.name:
        continue
    text = f.read_text(encoding="utf-8")
    sanskrit_by_id[gk.parse_verse_file(text).id] = gk.parse_word_meanings(text)

sanskrit_terms = gk.aggregate_sanskrit_terms(sanskrit_by_id)
run_ops(gk.sanskrit_term_constraint_ops())
run_ops(gk.sanskrit_term_ops(sanskrit_terms))
print(f"loaded {len(sanskrit_terms)} SanskritTerm nodes")

print("SanskritTerm nodes:", one("MATCH (t:SanskritTerm) RETURN count(t)"))
print("CONTAINS_TERM edges:", one("MATCH (:Verse)-[r:CONTAINS_TERM]->() RETURN count(r)"))
with driver.session(database=cfg.database) as s:
    rows = s.run(
        "MATCH (:Verse {id:'2.47'})-[:CONTAINS_TERM]->(t:SanskritTerm) "
        "RETURN t.lemma AS lemma ORDER BY lemma"
    ).value()
print("2.47 Sanskrit terms:", rows)


loaded 3340 SanskritTerm nodes
SanskritTerm nodes: 3340
CONTAINS_TERM edges: 7995
2.47 Sanskrit terms: ['adhikarah', 'akarmani', 'astu', 'bhuh', 'hetuh', 'kadachana', 'karma', 'ma', 'phaleshu', 'sangah']


In [10]:
# Enrich each verse with its Original Sanskrit and Transliteration
enrichment = {}
for f in sorted(VERSES_DIR.rglob("*.md")):
    if "Verse" not in f.name:
        continue
    text = f.read_text(encoding="utf-8")
    vid = gk.parse_verse_file(text).id
    enrichment[vid] = (gk.parse_sanskrit(text), gk.parse_transliteration(text))

run_ops(gk.verse_text_ops(enrichment))
print("sanskrit set:", one("MATCH (v:Verse) WHERE v.sanskrit <> '' RETURN count(v)"))
print("transliteration set:", one("MATCH (v:Verse) WHERE v.transliteration <> '' RETURN count(v)"))


sanskrit set: 701
transliteration set: 701


In [11]:
# 7. Theme layer (C1) — English-lemma themes over the Term layer
run_ops(gk.theme_constraint_ops())
run_ops(gk.theme_ops())
print("themes:", one("MATCH (t:Theme) RETURN count(t)"))
print("MENTIONS_THEME edges:", one("MATCH ()-[r:MENTIONS_THEME]->() RETURN count(r)"))


themes: 13
MENTIONS_THEME edges: 1286


In [ ]:
# 8. Concept layer — curated concepts grounded in the Sanskrit terms
run_ops(gk.concept_constraint_ops())
run_ops(gk.concept_ops())
run_ops(gk.concept_theme_alignment_ops())  # bridge Sanskrit concepts to English themes on shared names
print("concepts:", one("MATCH (c:Concept) RETURN count(c)"))
print("INSTANCE_OF edges:", one("MATCH (:SanskritTerm)-[r:INSTANCE_OF]->(:Concept) RETURN count(r)"))
print("EXPRESSES_CONCEPT edges:", one("MATCH (:Verse)-[r:EXPRESSES_CONCEPT]->() RETURN count(r)"))
print("ALIGNS_WITH edges:", one("MATCH (:Concept)-[r:ALIGNS_WITH]->(:Theme) RETURN count(r)"))


concepts: 22
INSTANCE_OF edges: 139
EXPRESSES_CONCEPT edges: 841


In [13]:
# 9. Character layer — the cast discovered from Word Meanings glosses
mentions = {}
for f in sorted(VERSES_DIR.rglob("*.md")):
    if "Verse" not in f.name:
        continue
    txt = f.read_text(encoding="utf-8")
    people = gk.people_referenced_in_glosses(gk.parse_word_meanings(txt), gk.PRINCIPALS)
    if people:
        mentions[gk.parse_verse_file(txt).id] = sorted(people)

run_ops(gk.character_constraint_ops())
run_ops(gk.character_ops(mentions))
print("characters:", one("MATCH (c:Character) RETURN count(c)"))
print("MENTIONS_CHARACTER edges:", one("MATCH ()-[r:MENTIONS_CHARACTER]->() RETURN count(r)"))


characters: 15
MENTIONS_CHARACTER edges: 294


## 10. Weapons & Vibhuti

Two small entity layers drawn straight from the Word Meanings: the six named
war-conches of Chapter 1 (`Conch`, with each warrior linked via `SOUNDS_CONCH`),
and the Chapter 10 verses where Krishna explicitly declares "I am …" (`asmi`),
grouped under a single `Vibhuti` node (`DECLARES_VIBHUTI`).


In [14]:
# Named war-conches (Ch.1 v15-16) and Krishna's explicit "I am" glories (Ch.10)
conch_mentions = {}
vibhuti_ids = []
for vid, rows in sanskrit_by_id.items():
    conches = sorted(gk.conches_in_glosses(rows))
    if conches:
        conch_mentions[vid] = conches
    if gk.is_vibhuti_declaration(int(vid.split(".")[0]), rows):
        vibhuti_ids.append(vid)

run_ops(gk.conch_constraint_ops())
run_ops(gk.conch_ops(conch_mentions))
run_ops(gk.vibhuti_constraint_ops())
run_ops(gk.vibhuti_ops(vibhuti_ids))
print("conches:", one("MATCH (c:Conch) RETURN count(c)"))
print("SOUNDS_CONCH edges:", one("MATCH (:Character)-[r:SOUNDS_CONCH]->(:Conch) RETURN count(r)"))
print("vibhuti declarations:", one("MATCH (:Verse)-[r:DECLARES_VIBHUTI]->(:Vibhuti) RETURN count(r)"))


conches: 6
SOUNDS_CONCH edges: 6
vibhuti declarations: 13


## 11. Semantic similarity (C2)

Pinned sentence-transformer embeddings on `Verse.translation`, a cosine vector
index, and calibrated canonical `SIMILAR_TO` edges. Requires the model
(`GITA_EMBEDDING_MODEL_PATH` for offline). The `SIMILAR_TO` load is gated behind
a manual `QUALITY_APPROVED` review.


In [15]:
import dataclasses
import hashlib

import numpy as np
import gita_embeddings as ge


def query_all(q, **p):
    with driver.session(database=cfg.database) as s:
        return [dict(r) for r in s.run(q, **p)]


emb_config = gk.EmbeddingConfig()
c2_rows = query_all(
    "MATCH (v:Verse) OPTIONAL MATCH (v)-[:MENTIONS_THEME]->(th:Theme) "
    "RETURN v.id AS id, v.chapter AS chapter, v.verse AS verse, "
    "v.translation AS translation, v.embedding AS embedding, "
    "v.embedding_model AS m, v.embedding_revision AS rev, "
    "v.embedding_dimension AS dim, v.embedding_input_sha256 AS h, "
    "collect(th.name) AS themes ORDER BY chapter, verse"
)


def _sha(t):
    return hashlib.sha256(t.encode()).hexdigest()


stale = [r for r in c2_rows
         if r["embedding"] is None or r["m"] != emb_config.model_id
         or r["rev"] != emb_config.revision or r["dim"] != emb_config.dimensions
         or r["h"] != _sha(r["translation"])]
print("stale embeddings:", len(stale), "/", len(c2_rows))
if stale:
    model = ge.load_embedding_model(emb_config)
    mat = ge.encode_translations(model, [r["translation"] for r in stale], emb_config.dimensions)
    persist = [{"id": stale[i]["id"], "embedding": mat[i].tolist(),
                "input_sha256": _sha(stale[i]["translation"])} for i in range(len(stale))]
    run_ops(gk.embedding_ops(persist, emb_config))
    print("persisted", len(persist), "embeddings")


stale embeddings: 701 / 701


/Users/akhilesh.koul/Documents/GitHub/CodePlayground/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 43660.96it/s]


Batches:   0%|          | 0/22 [00:00<?, ?it/s]

...


Batches:  77%|███████▋  | 17/22 [00:02<00:00, 12.29it/s]


Batches:  86%|████████▋ | 19/22 [00:02<00:00, 13.35it/s]


Batches:  95%|█████████▌| 21/22 [00:02<00:00, 14.40it/s]


Batches: 100%|██████████| 22/22 [00:02<00:00,  7.36it/s]

persisted 701 embeddings


In [16]:
# Assemble the (701, 768) matrix and (re)create the cosine vector index
ids = [r["id"] for r in c2_rows]
emb = {r["id"]: r["embedding"]
       for r in query_all("MATCH (v:Verse) RETURN v.id AS id, v.embedding AS embedding")}
embeddings = np.array([emb[i] for i in ids], dtype=np.float32)
assert embeddings.shape == (701, emb_config.dimensions), embeddings.shape
run_ops(gk.vector_index_ops(emb_config))
print("vector index issued; matrix", embeddings.shape)


vector index issued; matrix (701, 768)


In [17]:
# Exact cosine matrix, calibrate threshold against theme coverage, build canonical pairs
import os

similarity_matrix = embeddings @ embeddings.T
chapters = {r["id"]: r["chapter"] for r in c2_rows}
themes_by_id = {r["id"]: set(r["themes"]) - {None} for r in c2_rows}
stats = gk.evaluate_thresholds(
    ids, similarity_matrix, emb_config.top_k, gk.CANDIDATE_THRESHOLDS, chapters, themes_by_id
)
threshold = gk.select_similarity_threshold(stats, set(gk.THEMES))
final_config = dataclasses.replace(emb_config, threshold=threshold)
final_pairs = gk.build_similarity_pairs(ids, similarity_matrix, emb_config.top_k, threshold)
print(f"selected threshold {threshold}: {len(final_pairs)} canonical pairs")
# Set True here after eyeballing the pairs, or export GITA_APPROVE_SIMILARITY=1 for a headless run.
QUALITY_APPROVED = os.environ.get("GITA_APPROVE_SIMILARITY") == "1"


selected threshold 0.55: 2260 canonical pairs


In [18]:
# Manual quality gate: after reviewing the pairs above, set QUALITY_APPROVED = True and rerun this cell.
if not QUALITY_APPROVED:
    raise RuntimeError("quality gate: review pairs, set QUALITY_APPROVED = True, then rerun")
run_ops(gk.clear_similarity_ops())
run_ops(gk.similarity_ops(final_pairs, final_config))
print("loaded", len(final_pairs), "SIMILAR_TO edges")


loaded 2260 SIMILAR_TO edges


## 12. Close


In [19]:
driver.close()